In [0]:
from pyspark.sql import functions as F

print("Rebuilding Gold Macro Table with strict Multi-Column Join (Municipality + State)...")

# 1. Master Crime Index (Unique per Cleaned Municipality + State)
df_crime_agg = spark.read.table("silver_sinesp_crime_data_cleansed") \
    .filter(F.col("municipality_name_clean").isNotNull() & (F.col("municipality_name_clean") != "NAO INFORMADO")) \
    .groupBy("municipality_name_clean", "state_code") \
    .agg(
        F.max("municipality_name_original").alias("municipality_name_original"),
        F.sum("occurrences_count").alias("total_occurrences")
    )

# 2. Reading Dimensions and dropping duplicates to prevent row explosion
df_pop = spark.read.table("silver_ibge_population_cleansed") \
    .select("municipality_name_clean", "population_estimated") \
    .dropDuplicates(["municipality_name_clean"])

df_econ = spark.read.table("dim_municipality_economy_pib") \
    .select("municipality_name_clean", "gdp_per_capita") \
    .dropDuplicates(["municipality_name_clean"])

df_idh = spark.read.table("dim_municipality_idh") \
    .select("municipality_name_clean", "idh_composite") \
    .dropDuplicates(["municipality_name_clean"])

df_fleet = spark.read.table("silver_senatran_fleet_detailed") \
    .groupBy("municipality_name_clean") \
    .agg(F.sum("fleet_quantity").alias("fleet_total_count"))

# 3. Executing Safe Joins
df_macro = df_crime_agg \
    .join(df_pop, on="municipality_name_clean", how="left") \
    .join(df_econ, on="municipality_name_clean", how="left") \
    .join(df_fleet, on="municipality_name_clean", how="left") \
    .join(df_idh, on="municipality_name_clean", how="left")

# 4. Calculating Normalized Rates
df_macro_final = df_macro.withColumn(
    "crime_rate_per_100k",
    F.when(F.col("population_estimated") > 0, 
         F.round((F.col("total_occurrences") / F.col("population_estimated")) * 100000, 2))
    .otherwise(0.0)
).withColumn(
    "theft_rate_per_1k_fleet",
    F.when(F.col("fleet_total_count") > 0, 
         F.round((F.col("total_occurrences") / F.col("fleet_total_count")) * 1000, 2))
    .otherwise(0.0)
)

# 5. Materializing Gold Table
table_name = "gold_public_safety_macro"
df_macro_final.write.format("delta").mode("overwrite").saveAsTable(table_name)

# 6. Definitive Validation Check (Composite Key)
check = df_macro_final.groupBy("municipality_name_clean", "state_code").count().filter("count > 1")

if check.count() == 0:
    print("ABSOLUTE SUCCESS! Gold table is 100% integral and free of duplicates.")
else:
    print(f"Warning: There are still {check.count()} duplicate combinations.")

display(df_macro_final.limit(10))